# Day 4 — Fund Performance Analytics
**Bluestock Fintech Capstone | Mutual Fund Analytics Platform**

Covers all performance and risk metrics for 40 mutual fund schemes:
daily returns, CAGR, Sharpe, Sortino, Alpha, Beta, Max Drawdown, Fund Scorecard, and Benchmark comparison.


## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')   # remove this line when running in Jupyter locally
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

ROOT    = Path().resolve().parent
RAW     = ROOT / 'data' / 'raw'
REPORTS = ROOT / 'reports'
REPORTS.mkdir(parents=True, exist_ok=True)

plt.rcParams['figure.dpi']        = 120
plt.rcParams['font.family']       = 'DejaVu Sans'
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False

RF_ANNUAL = 0.065          # RBI repo rate proxy — 6.5%
RF_DAILY  = RF_ANNUAL / 252

print("Setup complete.")


## 2. Load Data

In [ ]:
# using raw NAV (business days only) for return calculations
# the forward-filled version adds weekends which would distort return distributions
nav_raw     = pd.read_csv(RAW / '02_nav_history.csv')
nav_raw['date'] = pd.to_datetime(nav_raw['date'])

bench_raw   = pd.read_csv(RAW / '10_benchmark_indices.csv')
bench_raw['date'] = pd.to_datetime(bench_raw['date'])

fund_master = pd.read_csv(RAW / '01_fund_master.csv')
fund_names  = fund_master.set_index('amfi_code')['scheme_name'].to_dict()
expense_map = fund_master.set_index('amfi_code')['expense_ratio_pct'].to_dict()

print(f"NAV         : {len(nav_raw):,} rows | {nav_raw['amfi_code'].nunique()} schemes")
print(f"Benchmarks  : {len(bench_raw):,} rows | indices: {sorted(bench_raw['index_name'].unique())}")
print(f"Date range  : {nav_raw['date'].min().date()} to {nav_raw['date'].max().date()}")


## 3. Daily Returns
`daily_return = (nav_t / nav_t-1) - 1`

Computed per scheme using pct_change() after sorting by date.


In [ ]:
nav = nav_raw.sort_values(['amfi_code', 'date']).copy()
nav['daily_return'] = nav.groupby('amfi_code')['nav'].pct_change()
returns = nav.dropna(subset=['daily_return']).copy()

print(f"Return rows : {len(returns):,}")
print()
print("Distribution summary (all funds):")
print(returns['daily_return'].describe().apply(lambda x: f"{x:.6f}"))

extreme = returns[returns['daily_return'].abs() > 0.10]
print(f"\nDays with |return| > 10%: {len(extreme)}")


In [ ]:
# return distribution for 4 sample funds
sample_codes = nav['amfi_code'].unique()[:4]

fig, axes = plt.subplots(2, 2, figsize=(12, 7))
fig.suptitle('Daily Return Distribution — Sample Funds', fontsize=14, fontweight='bold')

for ax, code in zip(axes.flat, sample_codes):
    data = returns[returns['amfi_code'] == code]['daily_return'] * 100
    ax.hist(data, bins=60, color='steelblue', edgecolor='white', linewidth=0.3, alpha=0.85)
    ax.axvline(data.mean(), color='tomato', linewidth=1.5, linestyle='--',
               label=f'Mean: {data.mean():.3f}%')
    ax.set_title(fund_names.get(code, str(code))[:45], fontsize=9)
    ax.set_xlabel('Daily Return (%)', fontsize=8)
    ax.set_ylabel('Frequency', fontsize=8)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig(REPORTS / 'daily_return_distribution.png', bbox_inches='tight')
plt.show()
print("Saved: daily_return_distribution.png")


## 4. CAGR — 1 Year and 3 Year
`CAGR = (NAV_end / NAV_start) ^ (1/n) - 1`

NAV data covers Jan 2022 to May 2026 (~4.4 years).
5yr CAGR is not computable with this dataset.


In [ ]:
END_DATE  = nav['date'].max()
START_1YR = END_DATE - pd.DateOffset(years=1)
START_3YR = END_DATE - pd.DateOffset(years=3)

nav_end_map = nav.groupby('amfi_code').apply(lambda g: g.sort_values('date').iloc[-1]['nav'])
cagr_records = []

for code in nav['amfi_code'].unique():
    nav_end = nav_end_map[code]

    s1 = nav[(nav['amfi_code']==code) & (nav['date'] <= START_1YR)].sort_values('date')
    s3 = nav[(nav['amfi_code']==code) & (nav['date'] <= START_3YR)].sort_values('date')

    n1 = s1.iloc[-1]['nav'] if len(s1) > 0 else np.nan
    n3 = s3.iloc[-1]['nav'] if len(s3) > 0 else np.nan

    c1 = round(((nav_end / n1) ** 1 - 1) * 100, 2)       if not pd.isna(n1) else np.nan
    c3 = round(((nav_end / n3) ** (1/3) - 1) * 100, 2)   if not pd.isna(n3) else np.nan

    cagr_records.append({
        'amfi_code'    : code,
        'scheme_name'  : fund_names.get(code, str(code)),
        'cagr_1yr_pct' : c1,
        'cagr_3yr_pct' : c3,
        'nav_latest'   : round(nav_end, 4),
    })

cagr_df = pd.DataFrame(cagr_records).sort_values('cagr_3yr_pct', ascending=False).reset_index(drop=True)
print("Top 10 by 3yr CAGR:")
print(cagr_df[['scheme_name','cagr_1yr_pct','cagr_3yr_pct']].head(10).to_string(index=False))


## 5. Sharpe Ratio
`Sharpe = (Rp - Rf) / Std(Rp) * sqrt(252)`

Rf = 6.5% annualised. A Sharpe > 1.0 is generally considered good.


In [ ]:
sharpe_records = []

for code, group in returns.groupby('amfi_code'):
    r         = group['daily_return']
    excess    = r - RF_DAILY
    ann_mean  = excess.mean() * 252
    ann_std   = r.std() * np.sqrt(252)
    sharpe    = ann_mean / ann_std if ann_std > 0 else np.nan

    sharpe_records.append({
        'amfi_code'      : code,
        'scheme_name'    : fund_names.get(code, str(code)),
        'ann_return_pct' : round(r.mean() * 252 * 100, 2),
        'ann_vol_pct'    : round(ann_std * 100, 2),
        'sharpe_ratio'   : round(sharpe, 4),
    })

sharpe_df = pd.DataFrame(sharpe_records).sort_values('sharpe_ratio', ascending=False).reset_index(drop=True)
sharpe_df['sharpe_rank'] = range(1, len(sharpe_df) + 1)

print(f"Sharpe range: {sharpe_df['sharpe_ratio'].min():.3f} to {sharpe_df['sharpe_ratio'].max():.3f}")
print()
print("Top 10 by Sharpe:")
print(sharpe_df[['scheme_name','ann_return_pct','ann_vol_pct','sharpe_ratio']].head(10).to_string(index=False))


## 6. Sortino Ratio
Same as Sharpe but denominator uses only **downside** std dev (negative return days).
Penalises bad volatility, not upside volatility.


In [ ]:
sortino_records = []

for code, group in returns.groupby('amfi_code'):
    r           = group['daily_return']
    excess_ann  = (r.mean() - RF_DAILY) * 252
    neg         = r[r < RF_DAILY] - RF_DAILY
    ds          = neg.std() * np.sqrt(252) if len(neg) > 1 else np.nan
    sortino     = excess_ann / ds if (ds and ds > 0) else np.nan

    sortino_records.append({
        'amfi_code'        : code,
        'scheme_name'      : fund_names.get(code, str(code)),
        'sortino_ratio'    : round(sortino, 4),
        'downside_vol_pct' : round(ds * 100, 2) if not pd.isna(ds) else np.nan,
    })

sortino_df = pd.DataFrame(sortino_records).sort_values('sortino_ratio', ascending=False).reset_index(drop=True)
print("Top 10 by Sortino:")
print(sortino_df[['scheme_name','sortino_ratio','downside_vol_pct']].head(10).to_string(index=False))


## 7. Alpha and Beta — OLS Regression vs Nifty 100
`fund_return = alpha_daily + beta * nifty100_return + error`

Beta=1 means fund moves exactly with market. Alpha > 0 means the fund manager added value.
Alpha is annualised by multiplying daily intercept by 252.


In [ ]:
nifty100 = bench_raw[bench_raw['index_name'] == 'NIFTY100'].copy().sort_values('date')
nifty100['nifty_return'] = nifty100['close_value'].pct_change()
nifty100 = nifty100.dropna(subset=['nifty_return'])[['date', 'nifty_return']]

ab_records = []

for code, group in returns.groupby('amfi_code'):
    merged = group[['date', 'daily_return']].merge(nifty100, on='date', how='inner')
    if len(merged) < 30:
        continue

    slope, intercept, r_val, p_val, se = stats.linregress(
        merged['nifty_return'], merged['daily_return']
    )

    ab_records.append({
        'amfi_code'   : code,
        'scheme_name' : fund_names.get(code, str(code)),
        'alpha_ann'   : round(intercept * 252 * 100, 4),
        'beta'        : round(slope, 4),
        'r_squared'   : round(r_val ** 2, 4),
        'p_value'     : round(p_val, 4),
        'n_obs'       : len(merged),
    })

ab_df = pd.DataFrame(ab_records).sort_values('alpha_ann', ascending=False).reset_index(drop=True)
ab_df['alpha_rank'] = range(1, len(ab_df) + 1)
ab_df.to_csv(REPORTS / 'alpha_beta.csv', index=False)

print("alpha_beta.csv saved.")
print(f"Beta range: {ab_df['beta'].min():.3f} to {ab_df['beta'].max():.3f}")
print()
print("Top 10 by Alpha:")
print(ab_df[['scheme_name','alpha_ann','beta','r_squared']].head(10).to_string(index=False))


## 8. Maximum Drawdown
`Max DD = min(NAV / running_max - 1)`

Worst peak-to-trough loss. A -25% drawdown means the fund fell 25% from its highest point.


In [ ]:
dd_records = []

for code, group in nav.groupby('amfi_code'):
    g = group.sort_values('date').copy()
    g['running_max'] = g['nav'].cummax()
    g['drawdown']    = g['nav'] / g['running_max'] - 1

    max_dd      = g['drawdown'].min()
    dd_end      = g.loc[g['drawdown'].idxmin(), 'date']
    peak_idx    = g.loc[:g['drawdown'].idxmin(), 'nav'].idxmax()
    dd_start    = g.loc[peak_idx, 'date']

    dd_records.append({
        'amfi_code'        : code,
        'scheme_name'      : fund_names.get(code, str(code)),
        'max_drawdown_pct' : round(max_dd * 100, 2),
        'dd_start'         : dd_start.date(),
        'dd_end'           : dd_end.date(),
        'dd_days'          : (dd_end - dd_start).days,
    })

dd_df = pd.DataFrame(dd_records).sort_values('max_drawdown_pct').reset_index(drop=True)
dd_df['dd_rank'] = range(1, len(dd_df) + 1)

print("10 funds with worst drawdown:")
print(dd_df[['scheme_name','max_drawdown_pct','dd_start','dd_end','dd_days']].head(10).to_string(index=False))


## 9. Fund Scorecard (Composite 0–100)
| Metric | Weight | Direction |
|---|---|---|
| 3yr CAGR | 30% | Higher is better |
| Sharpe | 25% | Higher is better |
| Alpha | 20% | Higher is better |
| Expense Ratio | 15% | Lower is better |
| Max Drawdown | 10% | Less negative is better |


In [ ]:
scorecard = cagr_df[['amfi_code','scheme_name','cagr_3yr_pct']].copy()
scorecard = scorecard.merge(sharpe_df[['amfi_code','sharpe_ratio']], on='amfi_code', how='left')
scorecard = scorecard.merge(ab_df[['amfi_code','alpha_ann']],        on='amfi_code', how='left')
scorecard = scorecard.merge(dd_df[['amfi_code','max_drawdown_pct']], on='amfi_code', how='left')
scorecard['expense_ratio_pct'] = scorecard['amfi_code'].map(expense_map)

n_ = len(scorecard)

def rank_to_score(rank_col, n):
    return ((n - rank_col) / (n - 1) * 100).round(2)

scorecard['rank_cagr3']   = scorecard['cagr_3yr_pct'].rank(ascending=False)
scorecard['rank_sharpe']  = scorecard['sharpe_ratio'].rank(ascending=False)
scorecard['rank_alpha']   = scorecard['alpha_ann'].rank(ascending=False)
scorecard['rank_expense'] = scorecard['expense_ratio_pct'].rank(ascending=True)
scorecard['rank_dd']      = scorecard['max_drawdown_pct'].rank(ascending=False)

scorecard['composite_score'] = (
    0.30 * rank_to_score(scorecard['rank_cagr3'],   n_) +
    0.25 * rank_to_score(scorecard['rank_sharpe'],  n_) +
    0.20 * rank_to_score(scorecard['rank_alpha'],   n_) +
    0.15 * rank_to_score(scorecard['rank_expense'], n_) +
    0.10 * rank_to_score(scorecard['rank_dd'],      n_)
).round(2)

scorecard = scorecard.sort_values('composite_score', ascending=False).reset_index(drop=True)
scorecard.index = scorecard.index + 1

out_cols = ['amfi_code','scheme_name','cagr_3yr_pct','sharpe_ratio','alpha_ann',
            'expense_ratio_pct','max_drawdown_pct','composite_score']
scorecard[out_cols].to_csv(REPORTS / 'fund_scorecard.csv', index=True, index_label='rank')

print("fund_scorecard.csv saved.")
print()
print("Top 10 funds:")
print(scorecard[out_cols].head(10).to_string())


In [ ]:
# scorecard bar chart — top 20
top20 = scorecard.head(20)

fig, ax = plt.subplots(figsize=(12, 8))
colors_ = plt.cm.RdYlGn(top20['composite_score'][::-1] / 100)
bars = ax.barh(top20['scheme_name'].str[:40][::-1], top20['composite_score'][::-1],
               color=colors_, edgecolor='white', linewidth=0.5)

for bar, score in zip(bars, top20['composite_score'][::-1]):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{score:.1f}', va='center', fontsize=8)

ax.set_xlabel('Composite Score (0-100)', fontsize=11)
ax.set_title('Fund Scorecard — Top 20\n(30% CAGR + 25% Sharpe + 20% Alpha + 15% Expense + 10% Drawdown)',
             fontsize=12, fontweight='bold')
ax.set_xlim(0, 110)
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(REPORTS / 'fund_scorecard_chart.png', bbox_inches='tight', dpi=150)
plt.show()
print("Saved: fund_scorecard_chart.png")


## 10. Benchmark Comparison — Top 5 Funds vs Nifty 50 & Nifty 100
All series normalised to 100 at the 3-year start date for direct comparison.

**Tracking Error** = std(fund_return - benchmark_return) x sqrt(252)


In [ ]:
end_d   = nav['date'].max()
start_d = end_d - pd.DateOffset(years=3)
top5    = scorecard['amfi_code'].head(5).tolist()
top5_names = {c: fund_names.get(c, str(c))[:35] for c in top5}

nifty50_s  = bench_raw[bench_raw['index_name'] == 'NIFTY50'].set_index('date')['close_value']
nifty100_s = bench_raw[bench_raw['index_name'] == 'NIFTY100'].set_index('date')['close_value']

fig, ax = plt.subplots(figsize=(14, 7))
colors  = ['#2196F3','#4CAF50','#FF9800','#9C27B0','#F44336']
te_records = []

for i, code in enumerate(top5):
    fn = nav[(nav['amfi_code']==code) & (nav['date']>=start_d)].set_index('date')['nav']
    if len(fn) < 10:
        continue
    ax.plot(fn.index, fn / fn.iloc[0] * 100, color=colors[i],
            linewidth=1.6, label=top5_names[code], alpha=0.9)

    fr  = fn.pct_change().dropna()
    br  = nifty100_s.pct_change().dropna()
    al  = fr.align(br, join='inner')
    te  = round((al[0] - al[1]).std() * np.sqrt(252) * 100, 2)
    te_records.append({'fund': top5_names[code], 'tracking_error_pct': te})

n50_  = nifty50_s[nifty50_s.index  >= start_d]
n100_ = nifty100_s[nifty100_s.index >= start_d]
ax.plot(n50_.index,  n50_  / n50_.iloc[0]  * 100, color='black', lw=2, ls='--', label='Nifty 50',  alpha=0.7)
ax.plot(n100_.index, n100_ / n100_.iloc[0] * 100, color='gray',  lw=2, ls=':',  label='Nifty 100', alpha=0.7)

ax.set_title('Top 5 Funds vs Benchmark — 3 Year\n(Normalised to 100 at start)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date', fontsize=11)
ax.set_ylabel('Normalised Value (Base = 100)', fontsize=11)
ax.legend(fontsize=9, loc='upper left')
ax.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.savefig(REPORTS / 'benchmark_comparison.png', bbox_inches='tight', dpi=150)
plt.show()

print("Saved: benchmark_comparison.png")
print()
print("Tracking Error vs Nifty 100 (annualised):")
for row in te_records:
    print(f"  {row['fund']}: {row['tracking_error_pct']}%")


## 11. Summary

In [ ]:
print("=" * 60)
print("DAY 4 SUMMARY")
print("=" * 60)
top   = scorecard.iloc[0]
ts    = sharpe_df.iloc[0]
ta    = ab_df.iloc[0]
wd    = dd_df.iloc[0]
print(f"Best overall (scorecard) : {top['scheme_name'][:55]}  [{top['composite_score']}]")
print(f"Highest Sharpe           : {ts['scheme_name'][:55]}  [{ts['sharpe_ratio']}]")
print(f"Highest Alpha            : {ta['scheme_name'][:55]}  [{ta['alpha_ann']}%]")
print(f"Worst drawdown           : {wd['scheme_name'][:55]}  [{wd['max_drawdown_pct']}%]")
print()
print("Files in reports/:")
for f in sorted(REPORTS.glob('*.*')):
    print(f"  {f.name}")
